# T19 — Time-Travel Snapshots

Generate monthly point-in-time snapshots with configurable growth, churn, and seasonality.

**What you'll learn:**
- Generate 12 months of evolving data
- Configure growth rates and seasonal spikes
- Produce partitioned DataFrames with `_snapshot_date`
- Write partitioned Parquet for Lakehouse queries

## 1. Generate 12-Month Snapshots

In [1]:
from sqllocks_spindle import RetailDomain, TimeTravelEngine, TimeTravelConfig

engine = TimeTravelEngine()
result = engine.generate(
    domain=RetailDomain(),
    config=TimeTravelConfig(
        months=12,
        start_date="2025-01-01",
        growth_rate=0.05,           # 5% monthly growth
        churn_rate=0.02,            # 2% monthly churn
        seasonality={11: 1.5, 12: 2.0},  # Holiday spike
        seed=42,
    ),
    scale="small",
)

print(result.summary())

Time-Travel Result
Domain: retail
Snapshots: 13

  Month           Date          Tables       Rows
  ----------------------------------------------
  Month 0        2025-01-01         9     21,750
  Month 1        2025-02-01         9     22,401
  Month 2        2025-03-01         9     23,071
  Month 3        2025-04-01         9     23,762
  Month 4        2025-05-01         9     24,475
  Month 5        2025-06-01         9     25,209
  Month 6        2025-07-01         9     25,967
  Month 7        2025-08-01         9     26,745
  Month 8        2025-09-01         9     27,549
  Month 9        2025-10-01         9     28,374
  Month 10       2025-11-01         9     29,933
  Month 11       2025-12-01         9     32,326
  Month 12       2026-01-01         9     33,298


## 2. Inspect Snapshots

In [2]:
for snap in result.snapshots:
    customer_count = snap.row_counts.get('customer', 0)
    order_count = snap.row_counts.get('order', 0)
    print(f"Month {snap.month_index:2d} ({snap.snapshot_date}): "
          f"{customer_count:,} customers, {order_count:,} orders")

Month  0 (2025-01-01): 1,000 customers, 5,000 orders
Month  1 (2025-02-01): 1,030 customers, 5,150 orders
Month  2 (2025-03-01): 1,061 customers, 5,304 orders
Month  3 (2025-04-01): 1,093 customers, 5,463 orders
Month  4 (2025-05-01): 1,126 customers, 5,627 orders
Month  5 (2025-06-01): 1,160 customers, 5,796 orders
Month  6 (2025-07-01): 1,195 customers, 5,970 orders
Month  7 (2025-08-01): 1,231 customers, 6,149 orders
Month  8 (2025-09-01): 1,268 customers, 6,334 orders
Month  9 (2025-10-01): 1,306 customers, 6,524 orders
Month 10 (2025-11-01): 1,377 customers, 6,883 orders
Month 11 (2025-12-01): 1,487 customers, 7,434 orders
Month 12 (2026-01-01): 1,532 customers, 7,657 orders


## 3. Growth Visualization

In [3]:
months = [s.month_index for s in result.snapshots]
customers = [s.row_counts.get('customer', 0) for s in result.snapshots]

print("Customer Growth Over 12 Months:")
for m, c in zip(months, customers):
    bar = '#' * (c // 20)
    print(f"  Month {m:2d}: {c:5,} {bar}")

print(f"\nNet growth: {customers[0]:,} -> {customers[-1]:,} "
      f"({(customers[-1]/customers[0] - 1)*100:.1f}%)")

Customer Growth Over 12 Months:
  Month  0: 1,000 ##################################################
  Month  1: 1,030 ###################################################
  Month  2: 1,061 #####################################################
  Month  3: 1,093 ######################################################
  Month  4: 1,126 ########################################################
  Month  5: 1,160 ##########################################################
  Month  6: 1,195 ###########################################################
  Month  7: 1,231 #############################################################
  Month  8: 1,268 ###############################################################
  Month  9: 1,306 #################################################################
  Month 10: 1,377 ####################################################################
  Month 11: 1,487 ##########################################################################
  Month 12: 1,532 ##########

## 4. Partitioned Output

In [4]:
partitioned = result.to_partitioned_dfs()

for table_name, df in partitioned.items():
    dates = df['_snapshot_date'].nunique()
    print(f"{table_name}: {len(df):,} total rows across {dates} snapshots")

print("\nSample with _snapshot_date column:")
partitioned['customer'][['customer_id', '_snapshot_date']].head(10)

customer: 15,866 total rows across 13 snapshots
address: 23,819 total rows across 13 snapshots
product_category: 743 total rows across 13 snapshots
product: 7,932 total rows across 13 snapshots
promotion: 3,162 total rows across 13 snapshots
store: 2,358 total rows across 13 snapshots
order: 79,291 total rows across 13 snapshots
order_line: 198,222 total rows across 13 snapshots
return: 13,467 total rows across 13 snapshots

Sample with _snapshot_date column:


,customer_id,_snapshot_date
0,1,2025-01-01
1,2,2025-01-01
2,3,2025-01-01
3,4,2025-01-01
4,5,2025-01-01
5,6,2025-01-01
6,7,2025-01-01
7,8,2025-01-01
8,9,2025-01-01
9,10,2025-01-01


## 5. Write to Parquet (Lakehouse Ready)

```python
for table_name, df in partitioned.items():
    df.to_parquet(f"/lakehouse/Files/snapshots/{table_name}.parquet")
```

## 6. CLI Equivalent

```bash
spindle time-travel retail --months 12 --output ./snapshots/ \
  --growth-rate 0.05 --churn-rate 0.02 --format parquet
```